# CLaRa Full-Paper Pipeline (Kaggle T4)

Notebook này có 2 flow:
- Flow 1: Train from scratch trên HotpotQA (Stage I + Stage II).
- Flow 2: Transfer learning từ Apple E2E pretrained (fine-tune TriviaQA + SQuAD).

Ghi chú thực tế cho T4: dùng HotpotQA thay cho Qwen synthetic; giảm `num_candidates`/`top_k` nếu thiếu VRAM.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1: Clone repo và cài đặt môi trường
# Chỉ cần chạy 1 lần. Sau đó RESTART KERNEL và bỏ qua cell này.
# ═══════════════════════════════════════════════════════════════════

!rm -rf introml-clara-implementation
!git clone -b main https://github.com/Duy-Tuyen/introml-clara-implementation.git
%cd introml-clara-implementation

!pip install -r requirements.txt -q
%run setup_env.py

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2: Xác nhận thư mục làm việc
# ═══════════════════════════════════════════════════════════════════
import os
import sys

repo_root = "/kaggle/working/introml-clara-implementation"
if os.path.isdir(repo_root):
    os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("CWD:", os.getcwd())

## Flow 1 — Train from scratch (HotpotQA)

### Stage I (SCP)

Paper dùng data synthesis (Qwen). Ở đây dùng HotpotQA (tested dataset) để chạy được trên T4.
Bạn có thể giảm `CLARA_N_TRAIN` nếu runtime quá lâu.

In [ ]:
!CLARA_DATASET=hotpotqa CLARA_N_TRAIN=8000 CLARA_N_VAL=500 CLARA_OUTPUT_DIR=/kaggle/working/clara-ckpts python -m scripts.train_stage1

### Stage II (End-to-End)

Differentiable retrieval + ST estimator.
T4-friendly defaults: `num_candidates=8`, `top_k=2` trong `configs/config.py`.

In [ ]:
!CLARA_DATASET=hotpotqa CLARA_STAGE1_DIR=/kaggle/working/clara-ckpts/stage1_ep1 CLARA_OUTPUT_DIR=/kaggle/working/clara-ckpts python -m scripts.train_stage2

## Flow 2 — Transfer learning từ Apple E2E

Bước 1: tải pretrained E2E.
Bước 2: convert weights sang format adapter (query/generator) của code hiện tại.
Bước 3: fine-tune Stage II trên TriviaQA + SQuAD và evaluate.

In [ ]:
import os
import subprocess

# 1) Download Apple E2E pretrained
subprocess.run([
    "python", "-m", "scripts.download_pretrained",
    "--repo", "apple/CLaRa-7B-E2E",
    "--out", "/kaggle/working/pretrained-e2e",
], check=True)

# 2) Convert to Stage II adapter format (query/generator)
subprocess.run([
    "python", "-m", "scripts.convert_apple_e2e",
    "--input", "/kaggle/working/pretrained-e2e",
    "--output", "/kaggle/working/pretrained-e2e-converted",
], check=True)

stage2_init = "/kaggle/working/pretrained-e2e-converted"

# 3) Fine-tune + Eval cho từng dataset
for ds in ["triviaqa", "squad"]:
    out_dir = f"/kaggle/working/clara-ckpts-ft-{ds}"
    env = os.environ.copy()
    env.update({
        "CLARA_DATASET": ds,
        "CLARA_STAGE1_DIR": "/kaggle/working/clara-ckpts/stage1_ep1",
        "CLARA_STAGE2_INIT": stage2_init,
        "CLARA_OUTPUT_DIR": out_dir,
    })
    subprocess.run(["python", "-m", "scripts.train_stage2"], check=True, env=env)

    env_eval = os.environ.copy()
    env_eval.update({
        "CLARA_STAGE2_DIR": f"{out_dir}/stage2_ep1",
        "CLARA_DATASET": ds,
    })
    subprocess.run(["python", "-m", "scripts.evaluate"], check=True, env=env_eval)